In [0]:
df = spark.table('workspace.default.olx_silver')
display(df)

Databricks visualization. Run in Databricks to view.

In [0]:
display(df.select("district").distinct())

For handling outliers I chose to use iqr with a factor of 3, because it deletes all anomalies and only few real ads that does not impact statistics. The minimum price is set to 100000 to start detecting outliers.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window

iqr_window = Window.partitionBy('district', 'size_category')
df = df.withColumn('size_category', 
        when(col('size_m2') <= 35, 'Smart')
        .when((col('size_m2') > 35) & (col('size_m2') <= 50), 'Standart 1 room')
        .when((col('size_m2') > 50) & (col('size_m2') <= 70), 'Standart 2 rooms')
        .when((col('size_m2') > 70) & (col('size_m2') <= 100), 'Standart 3 rooms')
        .when((col('size_m2') > 100) & (col('size_m2') <= 130), 'Standart 4 rooms')
        .otherwise('Luxury'))

df_outliers = df.withColumn("Q1", expr("percentile_approx(price_uah, 0.25)").over(iqr_window)) \
    .withColumn("Q3", expr("percentile_approx(price_uah, 0.75)").over(iqr_window)) \
    .withColumn("IQR", col("Q3") - col("Q1")) \
    .withColumn("upper_bound", col("Q3") + 3 * col("IQR")) \
    .withColumn("lower_bound", col("Q1") - 1.5 * col("IQR"))
display(df_outliers.select('upper_bound', 'lower_bound', 'size_category', 'district', 'IQR', 'Q3').distinct())

In [0]:
display(df_outliers.select('id', 'price_uah', 'clean_date', 'district', 'size_category', 'url').filter(((col('price_uah') > col('upper_bound')) & (col('price_uah') > 1000000)) | (col('price_uah') < col('lower_bound'))))

As a result, table contains only outliers.

In [0]:
df_clean = df_outliers.filter(
    ((col('price_uah') <= col('upper_bound')) | 
    (col('price_uah') <= 100000))
    &
    ((col('price_uah') > col('lower_bound')))
)
df_clean = df_clean.drop("Q1", "Q3", "IQR", "upper_bound", 'lower_bound')
display(df_clean)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

We can see that there are no outliers and all flats are in probable price range